In [21]:
from pathlib import Path
import pandas as pd
from ete3 import NCBITaxa
from Bio import Entrez, SeqIO
from collections import defaultdict
from functools import lru_cache
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import requests

In [22]:
DATA_PATH = 'data/m_rna_entities_to_be_processed.pkl'
mrna_entities = pd.read_pickle(DATA_PATH)

In [23]:
print("Unique mRNA entities:", len(mrna_entities))
print("NaN in Raw_ID_key:", mrna_entities['Raw_ID_key'].isna().sum())
print("NaN in Symbol:", mrna_entities['Symbol'].isna().sum())
print("Species involved:", mrna_entities['Species'].nunique())

Unique mRNA entities: 213723
NaN in Raw_ID_key: 0
NaN in Symbol: 1
Species involved: 111


In [29]:
db_counts = defaultdict(int)
for buckets in mrna_entities['ID_buckets']:
    for db, ids in buckets.items():
        if db != 'RNAcentral':  # exclude placeholder
            db_counts[db] += len(set(ids))  # count unique IDs per entity

# Print summary
for db, count in db_counts.items():
    print(f"{db}: {count} unique IDs across entities")


miRBase: 0 unique IDs across entities
NCBI: 202217 unique IDs across entities
Ensembl: 1177 unique IDs across entities
circBase: 0 unique IDs across entities
tRFdb: 0 unique IDs across entities
GenBank: 0 unique IDs across entities
Other: 0 unique IDs across entities


# Transcripts map

### 1- NCBI IDs

In [31]:
# --- 1) Load gene2refseq fast (vectorized) ---
g2r = pd.read_csv(
    "data/gene2refseq",
    sep="\t", comment="#", header=None,
    usecols=[0, 1, 2, 3, 15],   # tax_id, GeneID, status, RNA_accession, Symbol
    names=["taxid", "GeneID", "status", "rna_acc", "Symbol"], dtype=str, low_memory=False
)
# keep mRNA RefSeq
g2r = g2r[(g2r["status"] != "SUPPRESSED") & (g2r["rna_acc"] != "-") & (g2r["rna_acc"].str.startswith(("NM_", "XM_"), na=False))].copy()

In [32]:
mrna_entities

,Symbol,Category,Species,Raw_ID_key,RNA_IDs,ID_buckets
2635,0610009B22Rik,mRNA,10090,"('NCBI:66050',)",[NCBI:66050],"{'miRBase': [], 'NCBI': ['66050'], 'Ensembl': ..."
2637,0610010F05Rik,mRNA,10090,"('NCBI:71675',)",[NCBI:71675],"{'miRBase': [], 'NCBI': ['71675'], 'Ensembl': ..."
2638,0610010K14Rik,mRNA,10090,"('NCBI:104457',)",[NCBI:104457],"{'miRBase': [], 'NCBI': ['104457'], 'Ensembl':..."
2639,0610012D04Rik,mRNA,10090,"('NCBI:71676',)",[NCBI:71676],"{'miRBase': [], 'NCBI': ['71676'], 'Ensembl': ..."
2640,0610012G03Rik,mRNA,10090,"('NCBI:106264',)",[NCBI:106264],"{'miRBase': [], 'NCBI': ['106264'], 'Ensembl':..."
...,...,...,...,...,...,...
332905,zym1,mRNA,4896,"('NCBI:2541743',)",[NCBI:2541743],"{'miRBase': [], 'NCBI': ['2541743'], 'Ensembl'..."
332906,zyx,mRNA,7955,"('NCBI:564246',)",[NCBI:564246],"{'miRBase': [], 'NCBI': ['564246'], 'Ensembl':..."
332907,zzef1,mRNA,7955,"('NCBI:566682',)",[NCBI:566682],"{'miRBase': [], 'NCBI': ['566682'], 'Ensembl':..."
332908,zzef1,mRNA,8364,"('NCBI:100487543',)",[NCBI:100487543],"{'miRBase': [], 'NCBI': ['100487543'], 'Ensemb..."


In [15]:
# --- 2) Collapse to (taxid, GeneID) -> list of RefSeq mRNA transcripts --- SEARCHING JUST BY NCBI IDs AND SPECIES
refseq_map = (
    g2r.groupby(["taxid", "GeneID"])["rna_acc"]
      .agg(lambda s: sorted(set(s))).reset_index().rename(columns={"rna_acc": "refseq"}))

# --- 3) Merge onto your df_ncbi (Species == taxid) ---
df_ncbi["Species"] = df_ncbi["Species"].astype(str)
df_ncbi["GeneID"]  = df_ncbi["GeneID"].astype(str)

df_ncbi = df_ncbi.merge(refseq_map, left_on=["Species", "GeneID"], right_on=["taxid", "GeneID"], how="left").drop(columns=["taxid"])
df_ncbi.head()

,Symbol,Category,Species,Raw_ID_key,GeneID,refseq
0,0610009B22Rik,mRNA,10090,"(NCBI:66050,)",66050,"[NM_001417819.1, NM_001417820.1, NM_001417822...."
1,0610010F05Rik,mRNA,10090,"(NCBI:71675,)",71675,"[NM_001347242.1, NM_027860.3, XM_006514808.4, ..."
2,0610010K14Rik,mRNA,10090,"(NCBI:104457,)",104457,"[NM_001177601.1, NM_001177603.1, NM_001177606...."
3,0610012D04Rik,mRNA,10090,"(NCBI:71676,)",71676,NaN
4,0610012G03Rik,mRNA,10090,"(NCBI:106264,)",106264,[NM_134107.1]


In [14]:
print("Number of refseq:", df_ncbi["refseq"].notna().sum(),"/",len(df_ncbi), "Percentage:", 100*df_ncbi["refseq"].notna().sum()/len(df_ncbi))

Number of refseq: 194559 / 202217 Percentage: 96.21297912638403


### 2- 10.329 <NA> Symbol IDs (~4%)

In [150]:
# Obtain the cases without source identifiers (NaN in Raw_ID_key)
df_NA = mrna_entities[mrna_entities["Raw_ID_key"].isna()].copy()
df_NA.head(5)
# Here, just what the hell are that symbols

,Symbol,Category,Species,Raw_ID_key
192,14-3-3ÃÂ,mRNA,9606,<NA>
1092,18S_RRNA,mRNA,9606,<NA>
1865,28S_RRNA,mRNA,9606,<NA>
3125,7SLRNA,mRNA,9606,<NA>
47275,ArpH,mRNA,99287,<NA>


In [152]:
# 2) Build needed (taxid, Symbol) pairs
need = (df_NA[["Species", "Symbol"]].dropna().drop_duplicates().rename(columns={"Species": "taxid"}).astype(str))

# 3) Restrict gene2refseq to only needed pairs
g2r_need = g2r.merge(need, on=["taxid", "Symbol"], how="inner")

# 4) Build mapping (taxid, Symbol) -> list of mRNA RefSeq
refseq_df = (g2r_need.groupby(["taxid", "Symbol"])["rna_acc"].agg(lambda s: sorted(set(s))).reset_index(name="refseq"))

# 5) Merge back to df_NA
df_NA["Species"] = df_NA["Species"].astype(str)

df_NA = df_NA.merge(
    refseq_df,
    left_on=["Species", "Symbol"],
    right_on=["taxid", "Symbol"],
    how="left"
).drop(columns=["taxid"])

# 6) Report recovery
print("Number of refseq:", df_NA["refseq"].notna().sum(), "/", len(df_NA))

Number of refseq: 1649 / 10329


### 3-1.177 Ensemble IDs (~0.55%)

In [136]:
df_ENS = mrna_entities[mrna_entities["Raw_ID_key"].astype(str).str.contains("Ensembl:", na=False)].copy()
df_ENS["Ensembl_ID"] = (df_ENS["Raw_ID_key"].astype(str).str.extract(r"Ensembl:([^,\)]+)", expand=False))
print("Number of mRNA in Ensembl:", len(df_ENS))
df_ENS["Ensembl_ID"] = df_ENS["Ensembl_ID"].str.rstrip("'")
df_ENS.head()

## gene2refseq have some Ensembl IDs to try to retrieve refseq

Number of mRNA in Ensembl: 1177


,Symbol,Category,Species,Raw_ID_key,Ensembl_ID
934,1700094D03Rik,mRNA,10090,"(Ensembl:ENSMUSG00000078667,)",ENSMUSG00000078667
1111,2010015L04Rik,mRNA,10090,"(Ensembl:ENSMUSG00000042233,)",ENSMUSG00000042233
1534,2410017P09Rik,mRNA,10090,"(Ensembl:ENSMUSG00000073240,)",ENSMUSG00000073240
1752,2610524H06Rik,mRNA,10090,"(Ensembl:ENSMUSG00000092486,)",ENSMUSG00000092486
1856,2810453I06Rik,mRNA,10090,"(Ensembl:ENSMUSG00000048910,)",ENSMUSG00000048910


In [153]:
# 2) Build needed (taxid, Symbol) pairs
need = (df_ENS[["Species", "Symbol"]].dropna().drop_duplicates().rename(columns={"Species": "taxid"}).astype(str))

# 3) Restrict gene2refseq to only needed pairs
g2r_need = g2r.merge(need, on=["taxid", "Symbol"], how="inner")

# 4) Build mapping (taxid, Symbol) -> list of mRNA RefSeq
refseq_df = (g2r_need.groupby(["taxid", "Symbol"])["rna_acc"].agg(lambda s: sorted(set(s))).reset_index(name="refseq"))

# 5) Merge back to df_NA
df_ENS["Species"] = df_ENS["Species"].astype(str)

df_ENS = df_ENS.merge(
    refseq_df,
    left_on=["Species", "Symbol"],
    right_on=["taxid", "Symbol"],
    how="left"
).drop(columns=["taxid"])

# 6) Report recovery
print("Number of refseq:", df_ENS["refseq"].notna().sum(), "/", len(df_ENS))

Number of refseq: 213 / 1177


### Summary of results:
- mRNA-involved interactions: 9.224.109
- Unique mRNA entities: 213.723
### Source of the IDs | RefSeq Retrieved (for now and just using gene2refseq)
- NCBI       202.217  |       195.540    #(searching with IDs and symbols) 
- NA          10.329  |        1.649     #(searching just with symbol)
- Ensembl      1.177  |         213      #(searching just with symbol)  
- Total      213.723  |       197.402    #(92.36% of all the sequences has retrieved at least one refseq)

### For NCBI IDs: Number of refseq: 195.540 / 202.217 Percentage: ~96.7%

# GeneID → GO terms

In [44]:
# UPDATED code for parallelization

# MyGene.Info provides simple-to-use REST web services to query/retrieve gene annotation data. Find GO terms by NCBI GeneID.
# Im going to check for more options later, but for now I will try the /v3/gene endpoint with POST (robust for batches).
# Code trusting in GeneID.

# Standard annotation practice: GO terms are attached to the gene / gene product, not to each RefSeq transcript independently.


import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# --- Cache for GO terms ---
go_cache = {}  # GeneID -> {"BP": [...], "MF": [...], "CC": [...]}

def normalize_go(block):
    if not block:
        return []
    if isinstance(block, dict):
        block = [block]
    return [
        {"id": x.get("id"), "term": x.get("term"), "evidence": x.get("evidence")}
        for x in block
        if x.get("id") and x.get("term")
    ]

def fetch_go_batch(ids):
    """ Fetch GO annotations for a batch of GeneIDs using the batched POST endpoint. """
    if not ids:
        return
    # send batch request
    try:
        r = requests.post(
            "https://mygene.info/v3/gene",
            data={"ids": ",".join(map(str, ids)), "fields": "go"},
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=60
        )
        r.raise_for_status()
    except Exception as e:
        print(f"Batch request failed for {ids[:5]}..: {e}")
        return

    # populate cache
    resp = r.json()
    for rec in resp:
        # ensure we get the ID in a consistent key
        gid = str(rec.get("entrezgene") or rec.get("_id"))
        go = rec.get("go") or {}
        go_cache[gid] = {
            "BP": normalize_go(go.get("BP")),
            "MF": normalize_go(go.get("MF")),
            "CC": normalize_go(go.get("CC"))
        }

# --- Apply to DataFrame subset ---
print("Start:", datetime.now())
test_subset = df_ncbi_194k #.head(10000).copy()
unique_genes = [str(int(x)) for x in test_subset["GeneID"].dropna().unique()]

# --- Set up batch parameters ---
batch_size = 1000
batches = [unique_genes[i : i + batch_size] for i in range(0, len(unique_genes), batch_size)]

# Parallel batch processing
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(fetch_go_batch, batch) for batch in batches]
    for future in as_completed(futures):
        # no return value, just ensure completion
        future.result()

print("Fetched batches:", datetime.now())

# --- Map results back to dataframe ---
test_subset["GO"] = test_subset["GeneID"].map(
    lambda g: go_cache.get(str(int(g)), {"BP": [], "MF": [], "CC": []})
)
test_subset["GO_BP_count"] = test_subset["GO"].apply(lambda d: len(d["BP"]))
test_subset["GO_MF_count"] = test_subset["GO"].apply(lambda d: len(d["MF"]))
test_subset["GO_CC_count"] = test_subset["GO"].apply(lambda d: len(d["CC"]))

print("Done:", datetime.now())

Start: 2026-03-06 16:46:29.107820
Fetched batches: 2026-03-06 16:49:09.392913
Done: 2026-03-06 16:49:09.989914


In [45]:
print("GO BP terms counts:", test_subset["GO_BP_count"].sum())
print("GO MF terms counts:", test_subset["GO_MF_count"].sum())
print("GO CC terms counts:", test_subset["GO_CC_count"].sum())
# Numbers of refseq and GO terms are going to be crazy but important if we check correclty to be honest.

GO BP terms counts: 1003704
GO MF terms counts: 673342
GO CC terms counts: 730160
